In [ ]:
import anndata as ad
import pandas as pd
import numpy as np
from genome_tools.data.anndata import read_zarr_backed

from vinson.utils.data_formatting import extract_data_from_h5
from vinson.utils.helpers import read_configs
# Load the DHS dataset (original)
# dhs_file = "/net/seq/data2/projects/ENCODE4Plus/REGULOME/sequence_to_accessibility_model/training_data/NOV03/NOV03_3_epochs_with_weights_fc_squared_50.h5ad"
# dhs_adata = ad.read_h5ad(dhs_file)

# Load the variant dataset
variant_file = "/net/seq/data2/projects/sabramov/ENCODE4/ML/NOV17/variant_train_adata.h5ad"
variant_adata = ad.read_h5ad(variant_file)


In [ ]:
def extract_variant_data_from_anndata(train_adata: ad.AnnData, suffix: str):
    """
    Convert AnnData object to H5 format and extract embeddings.
    Args:
        adata (ad.AnnData): AnnData object containing DHS data.
        suffix (str): Suffix of the epoch/layer to extract. Gets added to layer names as `{layer}.{suffix}`. E.g. epoch_1, epoch_2, etc.
    
    Returns:
        data (dict): Dictionary containing extracted data arrays.
        embeddings_df (pd.DataFrame): DataFrame containing motif embeddings.
    """
    
    layers = {"ref_counts": None, "total_counts": None, "BAD": None, "logit_es": None}
    row_idx, col_idx = update_layers_dict_var(layers, train_adata, suffix)

    data = {
        'chrom': train_adata.var['#chr'].values[col_idx],
        'pos': train_adata.var['end'].values[col_idx],
        'ref': train_adata.var['ref'].values[col_idx],
        'alt': train_adata.var['alt'].values[col_idx],
        'sample_id': train_adata.obs_names[row_idx],
        'ref_counts': layers['ref_counts'].data,
        'total_counts': layers['total_counts'].data,
        'BAD': layers['BAD'].data,
        'logit_es': layers['logit_es'].data,
    }

    if 'indiv_id' in train_adata.obsm:
        data['indiv_id'] = get_indiv_id_info(train_adata, row_idx)
    return data

In [ ]:

def extract_data_from_train_anndata(train_adata: ad.AnnData, suffix: str):
    """
    Convert train AnnData object to H5 format and extract embeddings.
    Args:
        adata (ad.AnnData): AnnData object containing DHS data.
        suffix (str): Suffix of the epoch/layer to extract. Gets added to layer names as `{layer}.{suffix}`. E.g. epoch_1, epoch_2, etc.
    
    Returns:
        data (dict): Dictionary containing extracted data arrays.
        embeddings_df (pd.DataFrame): DataFrame containing motif embeddings.
    """
    
    layers = {"class": None, "density": None, "mean_bg_agg_cutcounts": None}

    #row_idx, col_idx, layers = update_layers_dict(layers, train_adata, suffix)
    row_idx, col_idx = update_layers_dict(layers, train_adata, suffix)

    data = {
        'read_depth': train_adata.obs['nuclear_reads'].values[row_idx],
        'sample_id': train_adata.obs_names[row_idx],
        'dhs_id': train_adata.var_names[col_idx],
        'chrom': train_adata.var['#chr'].values[col_idx],
        'summit': train_adata.var['dhs_summit'].values[col_idx],
        'background': layers['mean_bg_agg_cutcounts'].data,
        'class': layers['class'].data,
        'density': layers['density'].data,
    }
    if 'indiv_id' in train_adata.obsm:
        data['indiv_id'] = get_indiv_id_info(train_adata, row_idx)

    if 'dhs_weight' in train_adata.varm:
        data['dhs_weight'] = train_adata.varm['dhs_weight'][col_idx]

    data = sanitize_data(data)

    embeddings_df = train_adata.obsm['motif_embeddings']
    return data, embeddings_df

In [ ]:
def update_layers_dict(layers: dict, train_adata: ad.AnnData, suffix: str):
    assert len(layers) > 0, "Must provide at least one layer to extract"
    for layer_name in layers:
        epoch_layer_name = f"{layer_name}.{suffix}"
        layers[layer_name] = train_adata.layers[epoch_layer_name].tocoo()
    
    class_coo = layers["class"]
    row_idx, col_idx = class_coo.row, class_coo.col
    return row_idx, col_idx

In [ ]:
def get_indiv_id_info(train_adata: ad.AnnData, row_idx: np.ndarray):
    #return train_adata.obsm['indiv_id'].values[row_idx]
    return train_adata.obsm['indiv_id'][row_idx]


In [ ]:
def sanitize_data(data: dict, is_variant=False) -> dict:
    """Ensure that all data arrays are contiguous and correct dtype."""
    
    if is_variant:
        data_keys = {
            "chrom": np.str_,
            "pos": np.int32,
            "ref": np.str_,
            "alt": np.str_,
            "ref_counts": np.float32,
            "total_counts": np.float32,
            "BAD": np.float32,
            "sample_id": np.str_,
            "logit_es": np.float32,
        }
    else:
        data_keys = {
            "read_depth": np.float32,
            "sample_id": np.str_,
            "dhs_id": np.str_,
            "chrom": np.str_,
            "summit": np.int32,
            "background": np.float32,
            "class": np.int8,
            "density": np.float32,
        }

    optional_keys = {
        "indiv_id": np.str_,
        "dhs_weight": np.float32,
    }

    keys = {
        **data_keys,
        **{k: v for k, v in optional_keys.items() if k in data},
    }

    for key, dtype in keys.items():
        # Convert ANY incoming structure (Index, Series, list) into numpy array
        arr = np.asarray(data[key], dtype=object)

        if dtype == np.str_:
            mask = pd.isna(arr) | np.isin(arr, ['None', 'nan'])
            arr[mask] = ''

        data[key] = np.ascontiguousarray(arr.astype(dtype))

    if 'background' in data:
        data['background'] = np.nan_to_num(data['background'])

    return data


In [ ]:

def update_layers_dict_var(layers: dict, train_adata: ad.AnnData, suffix: str):
    assert len(layers) > 0, "Must provide at least one layer to extract"
    for layer_name in layers:
        epoch_layer_name = f"{layer_name}.{suffix}"
        layers[layer_name] = train_adata.layers[epoch_layer_name].tocoo()
    
    class_coo = layers["logit_es"]
    row_idx, col_idx = class_coo.row, class_coo.col
    return row_idx, col_idx


In [ ]:
data_out = extract_variant_data_from_anndata(variant_adata, 'epoch_1')

In [ ]:
from genome_tools import GenomicInterval, VariantInterval, df_to_variant_intervals
from genome_tools.data.extractors import FastaExtractor, TabixExtractor

from vinson.utils.sequence_utils import one_hot_encode, get_iupac_char_from_alleles
from vinson.utils.helpers import replace_at
import logging

genotype_extr: TabixExtractor = None
fasta_extr: FastaExtractor = None

In [ ]:
fasta_file = '/net/seq/data/genomes/human/GRCh38/noalts/GRCh38_no_alts.fa'

In [ ]:
if not fasta_extr:
    fasta_extr = FastaExtractor(fasta_file)

In [ ]:
genotype_file = '/net/seq/data2/projects/sabramov/ENCODE4/dnase-wasp.v5/output/all_variants_stats.bed.gz'

In [ ]:
genotype_file = '/net/seq/data2/projects/sabramov/ENCODE4/dnase-wasp.v5/phasing/output/all_phased.bed.gz'

In [ ]:
data = data_out
if genotype_file is not None:
    assert 'indiv_id' in data.keys(), "Sample to genotype mapping must include 'indiv_id' column."
    genotype_extr = None
    include_genotypes = True

In [ ]:
import gzip
if include_genotypes and not genotype_extr:
    with gzip.open(genotype_file, "rt") as f:
        phased = "phase_set" in f.readline()
        print('phased')
        print(phased)
        if phased:
            genotype_extr = TabixExtractor(
                    genotype_file,
                    skiprows=1,
                    columns=[
                        "chrom",
                        "start",
                        "end",
                        "ref",
                        "alt",
                        "indiv_id",
                        "gt",
                        "phase_block",
                    ],
                    na_values={"phase_block": "."},
                )
        else:
            print(f"[INFO] Using unphased genotype format ({genotype_file})")
            genotype_extr = TabixExtractor(
                genotype_file,
                columns=[
                    "chrom",
                    "start",
                    "end",
                    "rs_id",
                    "ref",
                    "alt",
                    "af_ref",
                    "af_alt",
                    "gt",
                    "_0",
                    "_1",
                    "_2",
                    "_3",
                    "indiv_id",
                ],
            )

In [ ]:
i = 0
chrom, pos, ref, alt = (
        data["chrom"][i],
        data["pos"][i],
        data["ref"][i],
        data["alt"][i],
    )

In [ ]:
#test specific
pos = 4104920
variant = GenomicInterval('chr19',pos,pos)
interval = variant.widen(1344 // 2)


In [ ]:
ref = 'G'
alt = 'A'

In [ ]:
rel_pos = pos - interval.start

In [ ]:
reference_variant=VariantInterval(
                chrom='chr19', start=pos, end=pos + 1, ref=ref, alt=alt
            )

In [ ]:
reference_variant

In [ ]:
seq = fasta_extr[interval]
seq_iupac = seq_ref = seq_alt = str(seq) # modify all 3 regardless


In [ ]:
variants = genotype_extr[interval]

In [ ]:
variants

In [ ]:
indiv_id = 'INDIV_0001'

In [ ]:
if variants["indiv_id"].str.endswith(".bed.gz").any():
    # DHS format
    key = f"{indiv_id}.bed.gz"
else:
    # variant format
    key = indiv_id

variants = variants[variants["indiv_id"] == key]

In [ ]:
extra_columns = ('gt',)
if "phase_set" not in variants.columns:
    if "phase_block" in variants.columns:
        variants = variants.rename(columns={"phase_block": "phase_set"})
    else:
        variants["phase_set"] = None
if reference_variant is not None:
    try:
        row = variants.set_index(["chrom", "start", "ref", "alt"]).loc[
            (reference_variant.chrom,
             reference_variant.start,
             reference_variant.ref,
             reference_variant.alt)
        ]
    except KeyError:
        raise ValueError(
            f"Query variant not found in genotyping file ")
    reference_variant.gt = row["gt"]
    phase_val = row.get("phase_set", None)
    reference_variant.phase_set = phase_val if not pd.isna(phase_val) and phase_val != "." else None

In [ ]:
variants

In [ ]:
if reference_variant is not None:
    # moved above: assert 'phase_set' in variants.columns, "Phased genotype data required for variant-aligned sequence extraction."
    try:
        # Look for the reference variant in the individual's genotypes
        phase_set = variants.set_index(
            ["chrom", "start", "ref", "alt"]
        ).loc[
            (
                reference_variant.chrom,
                reference_variant.start,
                reference_variant.ref,
                reference_variant.alt
            ),
            'phase_set'
        ]
        if phase_set is not None and phase_set != "." and not pd.isna(phase_set):
            reference_variant.phase_set = phase_set
            extra_columns = ('gt', 'phase_set')
    except KeyError:
        raise ValueError(
            "Query variant not found in genotyping file "
            f"({str(interval)}/{indiv_id}/{reference_variant.pos}/{reference_variant.alt})"
        )


In [ ]:
reference_variant.gt

In [ ]:
variants = df_to_variant_intervals(variants, extra_columns=extra_columns)
    

In [ ]:
variants_by_pos = {}
for v in variants:
    if v.start in variants_by_pos:
        variants_by_pos[v.start].append(v)
    else:
        variants_by_pos[v.start] = [v]


In [ ]:
variants_by_pos

In [ ]:
import warnings
ambiguous_positions = set()
for pos, vars_at_pos in variants_by_pos.items():
    rel_pos = pos - interval.start
    if len(vars_at_pos) > 1:
        # multiple variants at same position → warn & treat as unphased
        warnings.warn(
            f"Multiple variants at position {pos} for {indiv_id}: {[str(v) for v in vars_at_pos]}. "
            "Treating as unphased / ambiguous."
        )
        ambiguous_positions.add(pos)
        # combine alleles into IUPAC
        combined_ref = vars_at_pos[0].ref  # pick first as placeholder
        combined_alt = "".join(set(v.alt for v in vars_at_pos))
        base = get_iupac_char_from_alleles(combined_ref, combined_alt)
        seq_iupac = replace_at(seq_iupac, rel_pos, base)
        seq_ref = replace_at(seq_ref, rel_pos, combined_ref)
        seq_alt = replace_at(seq_alt, rel_pos, combined_alt)
    else:
        v = vars_at_pos[0]
        # phased variant logic
        if 'phase_set' in extra_columns and reference_variant is not None and reference_variant.phase_set == getattr(v, 'phase_set', None):
            base = get_iupac_char_from_alleles(v.ref, v.alt)
            seq_iupac = replace_at(seq_iupac, rel_pos, base)
            if v.gt == "1|0":
                seq_ref = replace_at(seq_ref, rel_pos, v.alt)
                seq_alt = replace_at(seq_alt, rel_pos, v.ref)
            elif v.gt == "0|1":
                seq_ref = replace_at(seq_ref, rel_pos, v.ref)
                seq_alt = replace_at(seq_alt, rel_pos, v.alt)
            else:
                raise ValueError(f'Phased genotype not recognized! {v}')
        else:
            # unphased logic
            assert v.gt[0] in ("0", "1") and v.gt[2] in ("0", "1"), f"Genotype format not recognized! {v} {v.gt}"
            variant_is_het = (v.gt[0] == "1" and v.gt[2] == "0") or (v.gt[0] == "0" and v.gt[2] == "1")
            if variant_is_het:
                base = get_iupac_char_from_alleles(v.ref, v.alt)
                seq_iupac = replace_at(seq_iupac, rel_pos, base)
                seq_ref = replace_at(seq_ref, rel_pos, v.ref)
                seq_alt = replace_at(seq_alt, rel_pos, v.alt)
            elif v.gt[0] == "1":
                base = v.alt
                seq_iupac = replace_at(seq_iupac, rel_pos, base)
                seq_ref = replace_at(seq_ref, rel_pos, base)
                seq_alt = replace_at(seq_alt, rel_pos, base)
            else:
                base = v.ref
                seq_iupac = replace_at(seq_iupac, rel_pos, base)
                seq_ref = replace_at(seq_ref, rel_pos, base)
                seq_alt = replace_at(seq_alt, rel_pos, base)


In [ ]:
reference_variant.phase_set

In [ ]:
interval.start

In [ ]:
reference_variant.gt

In [ ]:
if reference_variant is not None:
    #only for phased
    if phase_set is not None and phase_set != "." and not pd.isna(phase_set):
        if reference_variant.gt == "1|0":
            print('switching')
            seq_ref, seq_alt = seq_alt, seq_ref
    
    rel_pos = reference_variant.start - interval.start
    print(f'ref_relpos {seq_ref[rel_pos]} act {reference_variant.ref}, alt relpos {seq_alt[rel_pos]} act {reference_variant.alt}')
    print(rel_pos)
    if reference_variant.start not in ambiguous_positions:
        # only enforce ValueError if this position is NOT ambiguous
        if (seq_ref[rel_pos] != reference_variant.ref) or (seq_alt[rel_pos] != reference_variant.alt):
            raise ValueError(
                "Expected ref & alt alleles not found in correct position in sequences!",
                reference_variant, variants
            )